# Optional Extension: Scripted Intent and Shared Commitments

This ungraded extension is outside the required [five-notebook course](../notebook_manifest.yml). Complete the three labs and Capstone first; return to the [learner dashboard](../README.md#click-through-course-journey) for that route.

The example parses two scripted instructions, validates a `ProjectPlan`, and records a database-choice change in local shared memory. It uses ordinary Python functions and Pydantic; no provider key, CrewAI execution, extra package, file write, pull request, or deployment is involved. The historical NB10 identifier does not mean a tenth required lecture.


## What this notebook runs

```text
Scripted request + scripted change
                 |
                 v
Keyword parser -> Pydantic ProjectPlan
                 |
                 v
SharedMemory: latest database commitment
                 |
                 v
Python Coder / QA / Security placeholders
                 |
                 v
PullRequestSummary object printed locally
```

The QA function checks for string markers; it does not run pytest. The Security function records findings but currently returns `approved=True`. The human approval is hardcoded in the scripted session. A summary and an approval flag therefore provide no evidence that authentication is secure or software is ready to release. The `mock-jwt` code is a placeholder and must not be used as a real login implementation.


## Concept to inspect

An explicit shared commitment can carry an updated requirement to a downstream function. Observe how the database choice changes from PostgreSQL to MongoDB and how `coder_agent()` reads `SharedMemory.latest("db")`. The parser uses keyword rules rather than language-model reasoning; schema validation checks the plan's representation, not the quality of the requested design.


In [ ]:
from enum import Enum
from typing import Literal
from uuid import uuid4

from pydantic import BaseModel, ConfigDict, Field

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class AgentRole(str, Enum):
    CEO = "ceo"
    MANAGER = "manager"
    CODER = "coder"
    QA = "qa"
    SECURITY = "security"
    RELEASE = "release"

class ManagerInstruction(StrictModel):
    raw_text: str
    intent: Literal["create_project", "modify_constraint", "approve", "reject"]
    requested_by: AgentRole = AgentRole.CEO

class ProjectPlan(StrictModel):
    project_id: str = Field(default_factory=lambda: f"proj-{uuid4().hex[:6]}")
    goal: str
    api_style: Literal["REST", "GraphQL"] = "REST"
    database_choice: Literal["PostgreSQL", "MongoDB", "SQLite"]
    acceptance_criteria: list[str] = Field(min_length=1)
    risk_level: Literal["low", "medium", "high"]

class TeamCommitment(StrictModel):
    commitment_id: str = Field(default_factory=lambda: f"commit-{uuid4().hex[:6]}")
    owner: AgentRole
    key: str
    value: str
    reason: str

class HumanReviewDecision(StrictModel):
    status: Literal["approved", "rejected", "needs_changes"]
    reviewer: str
    notes: str

class CodePatch(StrictModel):
    files: dict[str, str]
    rationale: str

class TestResult(StrictModel):
    passed: bool
    log: str

class SecurityReview(StrictModel):
    approved: bool
    findings: list[str] = Field(default_factory=list)

class PullRequestSummary(StrictModel):
    title: str
    body: str
    database_choice: str
    approved_by_human: bool
    audit_trail: list[str]

class SharedMemory:
    def __init__(self):
        self.commitments: list[TeamCommitment] = []

    def add_commitment(self, commitment: TeamCommitment) -> None:
        self.commitments.append(commitment)

    def latest(self, key: str) -> TeamCommitment | None:
        matches = [item for item in self.commitments if item.key == key]
        return matches[-1] if matches else None

In [ ]:
def parse_manager_instruction(text: str) -> ManagerInstruction:
    lowered = text.lower()
    if "actually" in lowered or "instead" in lowered or "change" in lowered:
        intent = "modify_constraint"
    elif "approve" in lowered:
        intent = "approve"
    elif "reject" in lowered:
        intent = "reject"
    else:
        intent = "create_project"
    return ManagerInstruction(raw_text=text, intent=intent)

def mock_manager_llm_extract_plan(text: str) -> dict:
    """Teaching adapter: simulates an LLM translating natural language into a ProjectPlan candidate."""
    lowered = text.lower()
    if "database that doesn't exist" in lowered:
        database = "ImaginaryDB"  # Pydantic must reject this.
    elif "mongodb" in lowered:
        database = "MongoDB"
    elif "postgres" in lowered or "postgresql" in lowered:
        database = "PostgreSQL"
    else:
        database = "SQLite"

    return {
        "goal": text,
        "api_style": "GraphQL" if "graphql" in lowered else "REST",
        "database_choice": database,
        "acceptance_criteria": [
            "Expose user registration endpoint",
            "Expose login endpoint",
            "Persist users in the selected database",
            "Include tests and security review",
        ],
        "risk_level": "high" if "auth" in lowered or "secure" in lowered else "medium",
    }

def manager_create_plan(instruction: ManagerInstruction) -> ProjectPlan:
    return ProjectPlan.model_validate(
        mock_manager_llm_extract_plan(instruction.raw_text)
    )

def manager_apply_human_change(
    plan: ProjectPlan,
    instruction: ManagerInstruction,
    memory: SharedMemory,
) -> ProjectPlan:
    text = instruction.raw_text.lower()
    update = {}
    if "mongodb" in text:
        update["database_choice"] = "MongoDB"
        memory.add_commitment(
            TeamCommitment(
                owner=AgentRole.CEO,
                key="db",
                value="MongoDB",
                reason="Human CEO changed persistence requirement mid-workflow.",
            )
        )
    elif "postgres" in text:
        update["database_choice"] = "PostgreSQL"
        memory.add_commitment(
            TeamCommitment(
                owner=AgentRole.CEO,
                key="db",
                value="PostgreSQL",
                reason="Human CEO restored relational persistence requirement.",
            )
        )
    return plan.model_copy(update=update)

In [ ]:
def coder_agent(plan: ProjectPlan, memory: SharedMemory) -> CodePatch:
    db_commitment = memory.latest("db")
    database = db_commitment.value if db_commitment else plan.database_choice
    return CodePatch(
        files={
            "auth_api.py": (
                f"DATABASE = '{database}'\n"
                "def register_user(email, password):\n"
                "    return {'status': 'created', 'email': email}\n"
                "def login(email, password):\n"
                "    return {'token': 'mock-jwt'}\n"
            )
        },
        rationale=f"Implemented {plan.api_style} auth API using {database}.",
    )

def qa_agent(patch: CodePatch) -> TestResult:
    content = patch.files["auth_api.py"]
    required = ["register_user", "login", "DATABASE"]
    missing = [name for name in required if name not in content]
    return TestResult(
        passed=not missing,
        log="All auth API checks passed" if not missing else f"Missing: {missing}",
    )

def security_agent(plan: ProjectPlan, patch: CodePatch) -> SecurityReview:
    findings = []
    content = patch.files["auth_api.py"]
    if "password" not in content:
        findings.append("Authentication flow does not mention password handling.")
    if plan.risk_level == "high" and "mock-jwt" in content:
        findings.append("Teaching mock JWT must be replaced before production.")
    return SecurityReview(approved=True, findings=findings)

def release_manager(
    plan: ProjectPlan,
    patch: CodePatch,
    tests: TestResult,
    security: SecurityReview,
    human_decision: HumanReviewDecision,
    audit_trail: list[str],
) -> PullRequestSummary:
    return PullRequestSummary(
        title="Build user authentication API",
        body=(
            f"Goal: {plan.goal}\n"
            f"Database: {plan.database_choice}\n"
            f"Patch rationale: {patch.rationale}\n"
            f"QA: {tests.log}\n"
            f"Security findings: {security.findings}"
        ),
        database_choice=plan.database_choice,
        approved_by_human=human_decision.status == "approved",
        audit_trail=audit_trail,
    )

In [ ]:
def run_vibe_coding_session(scripted_inputs: list[str]) -> PullRequestSummary:
    memory = SharedMemory()
    audit_trail: list[str] = []

    create_instruction = parse_manager_instruction(scripted_inputs[0])
    plan = manager_create_plan(create_instruction)
    memory.add_commitment(
            TeamCommitment(
                owner=AgentRole.MANAGER,
                key="db",
                value=plan.database_choice,
                reason="Initial project plan from CEO instruction.",
            )
        )
    audit_trail.append(f"CEO request parsed into ProjectPlan using {plan.database_choice}")

    change_instruction = parse_manager_instruction(scripted_inputs[1])
    plan = manager_apply_human_change(plan, change_instruction, memory)
    audit_trail.append(f"CEO modified database choice to {plan.database_choice}")

    patch = coder_agent(plan, memory)
    tests = qa_agent(patch)
    security = security_agent(plan, patch)
    audit_trail.append(f"Coder produced patch: {patch.rationale}")
    audit_trail.append(f"QA result: {tests.log}")
    audit_trail.append(f"Security approved with findings: {security.findings}")

    human_decision = HumanReviewDecision(
        status="approved",
        reviewer="human-ceo",
        notes="Approved after confirming MongoDB requirement was honored.",
    )
    audit_trail.append(f"Human review status={human_decision.status}")

    return release_manager(
        plan=plan,
        patch=patch,
        tests=tests,
        security=security,
        human_decision=human_decision,
        audit_trail=audit_trail,
    )

summary = run_vibe_coding_session([
    "Build a REST API for user authentication with PostgreSQL",
    "Actually, use MongoDB instead",
])
print(summary.model_dump_json(indent=2))

assert summary.database_choice == "MongoDB"
assert summary.approved_by_human is True
assert any("modified database choice to MongoDB" in event for event in summary.audit_trail)

try:
    manager_create_plan(parse_manager_instruction("Build a REST API. Use a database that doesn't exist."))
    raise AssertionError("Invalid database should not validate.")
except Exception as exc:
    print(f"Validation blocked invalid CEO instruction: {type(exc).__name__}")

## Optional interface experiment

The supplied session uses scripted inputs. A later experiment could collect those inputs through a CLI and retain the same typed plans and commitments. A UI framework is not installed or required for this notebook.

The current parser checks `"approve"` before `"reject"`, and the session hardcodes approval rather than consuming a review instruction. Inspect those limitations before designing a human-review interface. The existing broad exception demonstration is illustrative output, not an independent pass/fail grader.


## Optional, ungraded design exercises

These are post-course extensions, not additional deliverables in the two-day workshop. Keep experiments in a separate copy of the notebook.

### Exercise 10.1: Inspect the translation layer

Compare `parse_manager_instruction()` with `manager_create_plan()`. List which scripted phrases the keyword rules support and which they misclassify. Specify separate cases for an invalid database, `"approve"`, `"reject"`, and an ambiguous instruction. A useful verification plan distinguishes `ValidationError` from unrelated exceptions.

### Exercise 10.2: Inspect a requirement change

Trace the database commitment written by `manager_apply_human_change()` and read by `coder_agent()`. Explain how the latest commitment changes the generated placeholder. Propose a check proving the downstream function used the updated commitment rather than the original request.

### Exercise 10.3: Design a release decision gate

The supplied session always creates an approved `HumanReviewDecision`, and `release_manager()` always returns a summary. Design explicit branches for `rejected` and `needs_changes`, and show where failed QA or Security findings would stop release eligibility. Do not treat the existing summary as a release gate.

A further workflow could connect these decisions to the [optional orchestration simulation](09_advanced_orchestration.ipynb). That connection is not implemented here. A complete delivery system would also need real test execution, security checks, controlled approval, and deployment evidence.
